# Public Portfolio Note

**Purpose:** Document the first-stage entity resolution workflow linking CMS hospital records to Google Places hospital records using ZIP3 blocking, RapidFuzz scoring, and ZIP5 geographic validation.

**Inputs:** CMS hospital quality files and a locally available Google Places API-derived hospital dataset with expected project column names.

**Outputs:** Sanitized matching workflow code and excluded intermediate match tables such as matched, manual-review, and unmatched queues.

**Public Data Limitation:** Raw Google API-derived records, row-level match previews, and manual Google place-id adjudication mappings are excluded from the public repository.


# Section 2: Entity Resolution & Fuzzy Matching (Stage 1)

This section documents the first-stage CMS-Google hospital matching workflow. It focuses on constrained candidate generation, probabilistic name scoring, and routing records for validation or follow-up.

**Key Steps:**
1. Load and standardize CMS and Google-derived hospital records.
2. Build ZIP3-blocked candidate pools to reduce cross-region false matches.
3. Score candidate names with RapidFuzz `token_set_ratio`.
4. Validate high-confidence matches with ZIP5 agreement.
5. Route ambiguous and unmatched records to review/follow-up queues.


## 2.1 Data Loading and Preparation

In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz, utils

In [ ]:

# Public path constants. These row-level files are excluded from the public repository.
CMS_HOSPITAL_GENERAL_PATH = "data/excluded/cms_hospital_general_information.csv"
CMS_HOSPITAL_VALUE_BASED_PURCHASING_PATH = "data/excluded/cms_value_based_purchasing_scores.csv"
GOOGLE_PLACES_HOSPITALS_PATH = "data/excluded/google_places_hospitals.csv"
MATCHED_HOSPITAL_CROSSWALK_PATH = "data/excluded/matched_hospital_crosswalk.csv"
UNMATCHED_HOSPITAL_REVIEW_QUEUE_PATH = "data/excluded/unmatched_hospital_review_queue.csv"
CMS_HOSPITAL_QUALITY_RECORDS_PATH = "data/excluded/cms_hospital_quality_records.csv"
GOOGLE_PLACES_CLEANED_RECORDS_PATH = "data/excluded/google_places_hospitals_cleaned.csv"


In [ ]:
def run_data_preparation_pipeline():
    """
    Loading datasets and prepare. 
    """
    print("Loading datasets.")
    
    df_base = pd.read_csv(CMS_HOSPITAL_GENERAL_PATH)
    df_scores = pd.read_csv(CMS_HOSPITAL_VALUE_BASED_PURCHASING_PATH)
    
    df_google = pd.read_csv(GOOGLE_PLACES_HOSPITALS_PATH, encoding='utf-8')

    # 1. Rename CMS columes, Turn CMS id into string
    for df in [df_base, df_scores]:
        df.columns = df.columns.str.replace(r'[ /]', '_', regex=True).str.lower()
        
    df_base['facility_id'] = df_base['facility_id'].astype(str).str.zfill(6)
    df_scores['facility_id'] = df_scores['facility_id'].astype(str).str.zfill(6)
    
    # 2. Rename Google columes
    df_google.columns = df_google.columns.str.lower()
    df_google = df_google.rename(columns={
        'place_id': 'google_id',
        'hospital_name': 'google_name',
        'city': 'google_city',
        'county': 'google_county', 
        'address': 'google_add', 
        'rating': 'google_star', 
        'rating_count': 'google_rating_count', 
    })

    # 3. Split Google address to get zipcode and state
    regex_pattern = r'\b([A-Z]{2})\s+(\d{5})(?:-?\d{4})?\b'
    extracted_cols = df_google['google_add'].str.extract(regex_pattern, expand=True)

    df_google['google_state'] = extracted_cols[0]
    df_google['google_zip'] = extracted_cols[1]
    
    # 4. Data cleaning: exclude non-CA samples
    df_google = df_google.dropna(subset=['google_state', 'google_zip']).copy()
    df_google = df_google[df_google['google_state'] == 'CA'].copy()

    # 5. Merge CMS Data and rename columns
    print("Merging CMS Data.")
    cols_base = ['facility_id', 'facility_name', 'county_parish', 'address', 'state', 'zip_code', 'hospital_overall_rating']
    cols_scores = ['facility_id', 'unweighted_normalized_clinical_outcomes_domain_score', 
                   'unweighted_person_and_community_engagement_domain_score', 'total_performance_score']
    
    df_cms = (
        df_base[df_base['state'] == 'CA'][cols_base]
        .merge(df_scores[cols_scores], on='facility_id', how='inner')
        .rename(columns={
            'facility_id': 'cms_id',
            'facility_name': 'cms_name',
            'county_parish': 'cms_county',
            'address': 'cms_add',
            'state': 'cms_state',
            'zip_code': 'cms_zip',
            'hospital_overall_rating': 'cms_star',
            'unweighted_normalized_clinical_outcomes_domain_score': 'clinical_score',
            'unweighted_person_and_community_engagement_domain_score': 'patient_exp_score',
            'total_performance_score': 'total_score'
        })
    )

    # 6. Turn CMS scores to numeric cols and turn missing data as NaN
    for col in ['cms_star', 'clinical_score', 'patient_exp_score', 'total_score']:
        df_cms[col] = pd.to_numeric(df_cms[col], errors='coerce')
        
    print("Finish. ")
    return df_cms, df_google

# Preparation
df_cms, df_google = run_data_preparation_pipeline()

print("-" * 50)
print("CMS:")
# Public copy: row-level preview/debug output removed.
pass
print("-" * 50)
print("Google:")
# Public copy: row-level preview/debug output removed.
pass


In [ ]:
# Public copy: df.info() output removed.
# Public copy: df.info() output removed.


In [ ]:
# df_google[df_google.duplicated()] # check duplicated samples

## 2.2 Fuzzy Matching

In [ ]:
def fuzzy_match_v3(df_cms, df_google, coarse_threshold=70):
    """
    Executes Stage 1 Entity Resolution utilizing Geographic Blocking and Fuzzy String Matching.
    
    Description:
    To optimize computational complexity and reduce false positives, the algorithm restricts 
    fuzzy comparisons within local 3-digit ZIP code pools (Geographic Blocking). It applies 
    a dual-validation logic: a probabilistic name match followed by a deterministic spatial match.
    
    Workflow:
    - Auto-Match: Name similarity >= threshold AND exact 5-digit ZIP match.
    - Dead Letter Queue (DLQ): Name similarity >= threshold BUT 5-digit ZIP mismatch (requires manual review).
    - Unmatched: No candidates meet the baseline similarity threshold.
    """
    print("Fuzzy matching initiated.")
    
    # 1. Spatial Feature Engineering: Extract 3-digit ZIP prefix for Geographic Blocking
    df_cms = df_cms.copy()
    df_google = df_google.copy()
    df_cms['zip3'] = df_cms['cms_zip'].astype(str).str[:3]
    df_google['zip3'] = df_google['google_zip'].astype(str).str[:3]
    
    # 2. Initialize routing queues
    auto_matched = []
    dlq = []
    unmatched = []
    
    # 3. Geographic Blocking: Restrict search space to zip-3 local pools
    for zip3, cms_group in df_cms.groupby('zip3'):
        local_pool = df_google[df_google['zip3'] == zip3]
        
        # If the local pool is empty, route all CMS facilities in this zip3 to 'unmatched'
        if local_pool.empty:
            for _, row in cms_group.iterrows():
                unmatched.append({
                    'cms_id': row['cms_id'], 
                    'cms_name': row['cms_name'], 
                    'cms_zip': row['cms_zip'], 
                    'cms_county': row['cms_county']
                })
            continue

        choices = local_pool['google_name_clean'].tolist() 
        
        # 4. Probabilistic Scoring within local pool
        for _, cms_row in cms_group.iterrows():
            matches = process.extract(
                query=cms_row['cms_name_clean'],
                choices=choices,
                scorer=fuzz.token_set_ratio, 
                limit=5, # Evaluate top 5 candidates to prevent occlusion 
                processor=utils.default_process
            )
            
            found = False
            best_mismatch = None
            cms_zip5 = str(cms_row['cms_zip'])[:5]
            
            for m in matches:
                clean_name, score, idx = m
                if score >= coarse_threshold:
                    google_row = local_pool.iloc[idx]
                    google_zip5 = str(google_row['google_zip'])[:5]
                    
                    # Construct matched record mapping
                    record = {
                        'cms_id': cms_row['cms_id'],
                        'google_id': google_row['google_id'],
                        'cms_zip': cms_zip5,
                        'google_zip': google_zip5,
                        'cms_name': cms_row['cms_name'],
                        'google_name': google_row['google_name'],
                        'cms_county': cms_row['cms_county'],
                        'google_county': google_row['google_county'],
                        'match_score': score,
                    }
                    
                    # 5. Deterministic Validation: Exact ZIP5 match guarantees facility identity
                    if cms_zip5 == google_zip5:
                        auto_matched.append(record)
                        found = True
                        break 
                        
                    # 6. Spatial Anomaly: High name similarity but ZIP5 discrepancy (Route to DLQ)
                    elif best_mismatch is None: 
                        record['anomaly_reason'] = 'ZIP5_MISMATCH_WITHIN_ZIP3'
                        best_mismatch = record

            # 7. Exception Routing: Route to DLQ if anomaly exists, otherwise to unmatched
            if not found:
                if best_mismatch:
                    dlq.append(best_mismatch)
                else:
                    unmatched.append({
                        'cms_id': cms_row['cms_id'], 
                        'cms_name': cms_row['cms_name'], 
                        'cms_zip': cms_zip5, 
                        'cms_county': cms_row['cms_county']
                    })
                    
    print("-" * 40)
    print(f"Auto-Matched (High Confidence): {len(auto_matched)}")
    print(f"Routed to DLQ (Spatial Anomaly): {len(dlq)}")
    print(f"Unmatched (Requires Stage 2 API): {len(unmatched)}")
    print("-" * 40)
    
    return pd.DataFrame(auto_matched), pd.DataFrame(dlq), pd.DataFrame(unmatched)


def apply_manual_fixes(df_matched, df_dlq, df_unmatched, fix_map, df_google):
    """
    Executes Stage 2 Manual Override and Exception Handling.
    
    Description:
    Processes expert-verified mappings (`fix_map`) to rescue false negatives from the DLQ 
    or unmatched queues. If the target facility lacks Google data, it tags the record 
    for Stage 2 secondary API scraping. Finally, it flushes the remaining unresolved DLQ 
    records into the unmatched pipeline.
    """
    for cms_id, google_id in fix_map.items():
        target_row = None
        
        # 1. Search and extract target facility from exception queues
        if cms_id in df_dlq['cms_id'].values:
            target_row = df_dlq[df_dlq['cms_id'] == cms_id].copy()
            df_dlq = df_dlq[df_dlq['cms_id'] != cms_id].copy()
        elif cms_id in df_unmatched['cms_id'].values:
            target_row = df_unmatched[df_unmatched['cms_id'] == cms_id].copy()
            df_unmatched = df_unmatched[df_unmatched['cms_id'] != cms_id].copy()
            
        # 2. Process manual assignment
        if target_row is not None and not target_row.empty:
            corrected_row = target_row.iloc[0:1].copy()
            g_match = df_google[df_google['google_id'] == google_id]
            
            if g_match.empty:
                # Target missing in Stage 1 data: Route to Stage 2 API scraper
                unmatched_subset = corrected_row[['cms_id', 'cms_name', 'cms_zip', 'cms_county']]
                df_unmatched = pd.concat([df_unmatched, unmatched_subset], ignore_index=True)
                print(f"Flagged for Stage 2 API retrieval: {cms_id}")
            else:
                # Deterministic Force-Match based on expert validation
                corrected_row['google_id'] = google_id
                corrected_row['match_score'] = 100.0
                corrected_row['anomaly_reason'] = 'MANUAL_VERIFIED'
                
                corrected_row['google_name'] = g_match['google_name'].values[0]
                corrected_row['google_zip'] = str(g_match['google_zip'].values[0])[:5]
                corrected_row['google_county'] = g_match['google_county'].values[0]
                
                df_matched = pd.concat([df_matched, corrected_row], ignore_index=True)
            
    # 3. Pipeline Flush: Push remaining unresolved DLQ records to Unmatched queue
    if not df_dlq.empty:
        unmatched_cols = ['cms_id', 'cms_name', 'cms_zip', 'cms_county']
        df_unmatched = pd.concat([df_unmatched, df_dlq[unmatched_cols]], ignore_index=True)
        df_dlq = pd.DataFrame(columns=df_dlq.columns) 
            
    return df_matched, df_dlq, df_unmatched

In [ ]:
# 1. set keywords
noise_keywords = 'fitness|wellness|pet|animal|vet|veterinary|dental|rehab|massage|spa' # not hospital
stop_words_pattern = r'\b(hospital|hosp|medical|med|center|ctr|clinic|emergency|services|health|hlth|dignity|foundation)\b' # frequent word affect fuzzy match score
advanced_noise = 'office|group|clinic|behavioral|sports|imaging|specialty|primary|urgent|student' # same hospital but with several google spot

# 2. clean names
df_cms['cms_name_clean'] = df_cms['cms_name'].str.lower().str.replace(stop_words_pattern, '', regex=True)
df_google_hospital= df_google[~df_google['google_name'].str.lower().str.contains(noise_keywords, na=False)].copy()
df_google_hospital['google_name_clean'] = df_google_hospital['google_name'].str.lower().str.replace(stop_words_pattern, '', regex=True)
df_google_hospital_clean = df_google_hospital[
    ~df_google_hospital['google_name'].str.lower().str.contains(advanced_noise, na=False)
].copy()

# 3. for system hospitals: same zipcode including some office/speciality, choose the one with most reviews
df_google_hospital_clean = df_google_hospital_clean.sort_values('google_rating_count', ascending=False).drop_duplicates(subset=['google_zip', 'google_name_clean'])

In [ ]:
# fuzzy match stage 1
df_matched, df_dlq, df_unmatched = fuzzy_match_v3(df_cms, df_google_hospital_clean, coarse_threshold=75)

# check match info
# Public copy: df.info() output removed.
# check duplicates
df_duplicates = df_matched[df_matched.duplicated(subset=['google_name', 'google_zip'], keep=False)].sort_values(by='google_name')

print(f"Duplication: {len(df_duplicates)} ")
if not df_duplicates.empty:
    # Public copy: row-level preview/debug output removed.
    pass
    
# check samples that zipcode only first 3 digits matched
if not df_dlq.empty:
    # Public copy: row-level preview/debug output removed.
    pass


## 2.3 Fixed some pairs manually

In [ ]:
# match zip-3 group and check mannually
df_dlq

In [ ]:
# Public copy: manual adjudication mappings are omitted because they contain
# Google API-derived place identifiers and row-level match decisions.
# In the private workflow, this dictionary maps selected CMS IDs to
# validated Google place IDs after manual review.
manual_fix_dict = {}

df_matched, df_dlq, df_unmatched = apply_manual_fixes(
    df_matched, df_dlq, df_unmatched, manual_fix_dict, df_google
)

print(f"Matched: {len(df_matched)}")
print(f"DLQ: {len(df_dlq)}")
print(f"Unmatched: {len(df_unmatched)}")


## 2.4 Output: csv file

In [ ]:
# Public copy: row-level/intermediate CSV exports are disabled.
# In the original private workflow, this notebook produced:
#
# - matched_hospital_crosswalk.csv
#   Matched CMS-Google hospital crosswalk used by the dataset merge notebook.
#
# - unmatched_hospital_review_queue.csv
#   Ambiguous or unresolved candidate matches for manual review / targeted API follow-up.
#
# - cms_hospital_quality_records.csv
#   CMS hospital quality records after column standardization for matching.
#
# - google_places_hospitals_cleaned.csv
#   Google Places-derived hospital records after column standardization for matching.
#
# These files are excluded from the public repository because they contain
# row-level Google Places API-derived records and intermediate matching artifacts.
#
# Private workflow only:
# df_matched.to_csv(MATCHED_HOSPITAL_CROSSWALK_PATH, index=False, encoding='utf-8-sig')
# df_unmatched.to_csv(UNMATCHED_HOSPITAL_REVIEW_QUEUE_PATH, index=False, encoding='utf-8-sig')
# df_cms.to_csv(CMS_HOSPITAL_QUALITY_RECORDS_PATH, index=False, encoding='utf-8-sig')
# df_google.to_csv(GOOGLE_PLACES_CLEANED_RECORDS_PATH, index=False, encoding='utf-8-sig')

print("Public copy: CSV exports disabled. Expected private outputs are documented above.")
